# 04 — Weights and pooling

Separate forecast weights, instrument weights, diversification multipliers, estimation, and pooled evidence.

## Weights and fitting

So far every weight was fixed. This notebook turns the estimation machinery
on — for real, over the historically eligible universe — and answers the two questions that
matter most in practice:

1. **Which optimiser?** The registry contains exactly four:
   `equal_weights`, `shrinkage`, `handcraft` (the default: hierarchical
   clustering into pairs + a Sharpe-based tilt), and `one_period`.
   (Older docs mention bootstrapping; it is no longer in the code.)
2. **Fit across or within instruments?** Five independent pooling switches
   control whether estimation shares information across instruments:

| config key | pools what |
|---|---|
| `forecast_scalar_estimate.pool_instruments` | scalar from cross-sectional absolute forecasts |
| `forecast_correlation_estimate.pool_instruments` | rule correlations |
| `forecast_weight_estimate.pool_gross_returns` | rule gross returns |
| `forecast_cost_estimates.use_pooled_costs` | full rule SR-cost summaries |
| `forecast_cost_estimates.use_pooled_turnover` | rule turnovers |

Pooling matters enormously here: half the Chinese universe has under ten
years of history — fitting rule weights on three years of one instrument's
returns is mostly fitting noise.

The actual defaults are deliberately mixed: scalars, correlations, gross
rule returns and turnover are pooled; **full costs are not**. This preserves
each contract's cost per trade while borrowing the rule's more stable
turnover estimate. "Pool everything" is neither the default nor good advice.

One wiring rule to burn in: **`config.instrument_weights` beats
`config.instruments`** when the system decides its universe. For estimated
runs you must *not* set `instrument_weights`, and must set `instruments`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import research as R

R.set_notebook_style()

In [ ]:
R.limit_blas_threads()   # ~90x90 matrix ops: OpenBLAS threading is pure overhead

from sysdata.config.configdata import Config
from sysdata.sim.db_futures_sim_data import dbFuturesSimData
from systems.accounts.accounts_stage import Account
from systems.basesystem import System
from systems.forecast_combine import ForecastCombine
from systems.forecast_scale_cap import ForecastScaleCap
from systems.forecasting import Rules
from systems.positionsizing import PositionSizing
from systems.rawdata import RawData

data = dbFuturesSimData()
universe = R.chinese_universe(data)
held_volume = R.held_contract_volumes(data, universe)
liquidity = R.liquidity_eligibility(held_volume, force_terminal_close=True)
ever_eligible_universe = list(liquidity.columns[liquidity.any(axis=0)])
never_eligible = sorted(set(universe) - set(ever_eligible_universe))
assert len(universe) == 95
assert len(ever_eligible_universe) == 93
assert never_eligible == ["CZCE_LR", "CZCE_PM"]

EWMAC = "systems.provided.rules.ewmac.ewmac"
EWMAC_DATA = ["rawdata.get_daily_prices", "rawdata.daily_returns_volatility"]
CARRY = "systems.provided.rules.carry.carry"
trading_rules = {
    "ewmac16_64": dict(function=EWMAC, data=EWMAC_DATA,
                       other_args=dict(Lfast=16, Lslow=64)),
    "ewmac32_128": dict(function=EWMAC, data=EWMAC_DATA,
                        other_args=dict(Lfast=32, Lslow=128)),
    "ewmac64_256": dict(function=EWMAC, data=EWMAC_DATA,
                        other_args=dict(Lfast=64, Lslow=256)),
    "carry10": dict(function=CARRY, data=["rawdata.raw_carry"],
                    other_args=dict(smooth_days=10)),
    "carry60": dict(function=CARRY, data=["rawdata.raw_carry"],
                    other_args=dict(smooth_days=60)),
    "carry125": dict(function=CARRY, data=["rawdata.raw_carry"],
                     other_args=dict(smooth_days=125)),
}
fixed_scalars = dict(ewmac16_64=3.75, ewmac32_128=2.65, ewmac64_256=1.87,
                     carry10=27.82, carry60=28.40, carry125=29.37)
print(f"{len(universe)} stored histories; {len(ever_eligible_universe)} ever "
      f"eligible for fitting; never eligible: {never_eligible}; "
      f"{len(trading_rules)} rules; "
      f"latest eligible: {int(liquidity.iloc[-1].sum())}")

## The experimental grid

One helper builds a fresh system per variant (estimation results are cached
*inside* a system, so reusing one across config changes is a classic bug),
extracts what we need, and frees the memory. Estimated variants switch on
the whole stack: forecast scalars, forecast weights + FDM, instrument
weights + IDM. `PointInTimePortfolios` replaces only the native portfolio
stage: it masks and renormalises either fixed or fitted weights using the
declared dated eligibility panel; every upstream forecast and fit remains
native. The only non-causal boundary is stated explicitly: if one of the
known ended histories is still eligible, we place a terminal zero target
early enough for the final stored quote to close it. That ex-post close is a
data-end convention, not a live delisting forecast.

All 95 histories remain in the eligibility audit. The native optimiser cannot
fit `CZCE_LR` or `CZCE_PM`: neither ever passes the rule and neither supplies a
usable cheap-rule account. We therefore omit only those two all-zero columns
from every optimiser variant. This is an ex-post computational omission, but
it cannot change portfolio membership or P&L because their dated weights are
zero on every observation.

In [ ]:
import copy, gc
import yaml
from IPython.utils.io import capture_output

with open(R.REPO_ROOT / "sysdata/config/defaults.yaml") as handle:
    DEFAULTS = yaml.safe_load(handle)

def estimate_block(name, **overrides):
    block = copy.deepcopy(DEFAULTS[name])
    block.update(overrides)
    return block

def make_system(**config_overrides):
    settings = dict(
        trading_rules=trading_rules,
        instruments=ever_eligible_universe,
        notional_trading_capital=100_000_000,
        base_currency="CNH",
        vol_normalise_currency_costs=False,
    )
    settings.update(config_overrides)
    return System(
        [
            Account(),
            R.PointInTimePortfolios(liquidity),
            PositionSizing(),
            RawData(),
            ForecastCombine(),
            ForecastScaleCap(),
            Rules(),
        ],
        dbFuturesSimData(),
        Config(settings),
    )

def run_variant(label, **config_overrides):
    system = make_system(**config_overrides)

    # The optimisers print progress bars for every fit. Keep the saved output
    # to one result line rather than thousands of terminal redraws.
    with capture_output():
        portfolio = system.accounts.portfolio()
        extracted = dict(
            label=label,
            returns=portfolio.percent.as_ts,
            stats=R.stats_row(portfolio, label),
            forecast_weights_rb=system.combForecast.get_forecast_weights("SHFE_RB"),
            forecast_weights_lc=system.combForecast.get_forecast_weights("GFEX_LC"),
            idm=system.portfolio.get_instrument_diversification_multiplier(),
            fdm_rb=system.combForecast.get_forecast_diversification_multiplier(
                "SHFE_RB"),
            instrument_weights=system.portfolio.get_instrument_weights(),
            scalar_carry60_rb=system.forecastScaleCap.get_forecast_scalar(
                "SHFE_RB", "carry60"),
        )
    del system
    gc.collect()
    print(f"[{label}] done; performance is compared later on common dates")
    return extracted

ESTIMATE_ALL = dict(
    use_forecast_scale_estimates=True,
    use_forecast_weight_estimates=True,
    use_forecast_div_mult_estimates=True,
    use_instrument_weight_estimates=True,
    use_instrument_div_mult_estimates=True,
)
pooled_scalar_without_backfill = estimate_block(
    "forecast_scalar_estimate", pool_instruments=True, backfill=False)

### Variant A — everything fixed (the baseline)

Fixed scalars, equal forecast weights within style buckets, equal
instrument weights, IDM pinned at its cap.

In [ ]:
fixed_forecast_weights = {rule: 0.5 / 3 for rule in
                          ["ewmac16_64", "ewmac32_128", "ewmac64_256"]}
fixed_forecast_weights.update({rule: 0.5 / 3 for rule in
                               ["carry10", "carry60", "carry125"]})
variant_a = run_variant(
    "A fixed",
    forecast_scalars=fixed_scalars,
    forecast_weights=fixed_forecast_weights,
    forecast_div_multiplier=1.5,
    instrument_weights={code: 1 / len(ever_eligible_universe)
                        for code in ever_eligible_universe},
    instrument_div_multiplier=2.5,
)

### Variant B — handcraft with the repo's mixed pooling defaults

Everything estimated. Scalars, forecast correlations, gross rule returns and
turnover pool across compatible instruments; full cost summaries remain
instrument-specific. Expanding windows, weekly fits. This is the slow honest
machinery—expect a long runtime on 93 historically eligible histories. We make one explicit
causality correction to the defaults: scalar `backfill=False`, so the first
future scalar estimate is not copied into the early sample.

In [ ]:
variant_b = run_variant(
    "B handcraft pooled",
    **ESTIMATE_ALL,
    forecast_scalar_estimate=pooled_scalar_without_backfill,
)

### Variant C — shrinkage instead of handcraft

Same estimation stack, but weights come from a shrunk mean-variance
optimisation (correlations and Sharpe ratios shrunk toward priors).
Note a defaults-file trap: `instrument_weight_estimate.shrinkage_mean`
is dead config — the live parameter is `shrinkage_SR`.

In [ ]:
variant_c = run_variant(
    "C shrinkage pooled",
    **ESTIMATE_ALL,
    forecast_scalar_estimate=pooled_scalar_without_backfill,
    forecast_weight_estimate=estimate_block("forecast_weight_estimate",
                                            method="shrinkage"),
    instrument_weight_estimate=estimate_block("instrument_weight_estimate",
                                              method="shrinkage"),
)

### Diagnostic D — what breaks when each instrument is fitted alone

This uses B's optimiser but fits scalars, rule correlations and rule gross
returns only on each instrument's own history. It deliberately does **not**
claim to be a portfolio variant: on the current universe a fully unpooled
portfolio is undefined.

The diagnostic asks for weights on veteran SHFE rebar, newer GFEX lithium,
and short-history GFEX palladium. Cost per trade remains market-specific while
turnover remains pooled, matching the repository defaults. The palladium call
may still lack enough annual boundaries to construct a first unpooled fit;
that outcome is caught and printed rather than assumed or allowed to crash
the notebook.

In [ ]:
unpooled_settings = dict(
    **ESTIMATE_ALL,
    forecast_scalar_estimate=estimate_block("forecast_scalar_estimate",
                                            pool_instruments=False,
                                            backfill=False),
    forecast_correlation_estimate=estimate_block(
        "forecast_correlation_estimate", pool_instruments=False),
    forecast_weight_estimate=estimate_block("forecast_weight_estimate",
                                            pool_gross_returns=False),
)
system_d = make_system(**unpooled_settings)
with capture_output():
    weights_d_rb = system_d.combForecast.get_forecast_weights("SHFE_RB")
    weights_d_lc = system_d.combForecast.get_forecast_weights("GFEX_LC")
    scalar_d_rb = system_d.forecastScaleCap.get_forecast_scalar(
        "SHFE_RB", "carry60")
    try:
        system_d.combForecast.get_forecast_weights("GFEX_PD")
    except (IndexError, ValueError) as error:
        newborn_failure = f"{type(error).__name__}: {error}"
    else:
        newborn_failure = None
del system_d
gc.collect()
variant_d = dict(
    forecast_weights_rb=weights_d_rb,
    forecast_weights_lc=weights_d_lc,
    scalar_carry60_rb=scalar_d_rb,
)
print("GFEX_PD unpooled fit:", newborn_failure or "now has enough history")

### Variant E — handcraft pooled, rolling **weight fits**

Only the forecast-weight and instrument-weight optimisers use the 20-year
rolling window here. Scalars and correlation estimates retain their default
expanding histories. Most Chinese histories are shorter than 20 years, and
even the veterans may have less than 20 usable years after all stages align.
This isolates whether the default rolling **weight** fit differs here; it is
not a completely rolling estimation stack.

In [ ]:
variant_e = run_variant(
    "E rolling weight fits",
    **ESTIMATE_ALL,
    forecast_scalar_estimate=pooled_scalar_without_backfill,
    forecast_weight_estimate=estimate_block("forecast_weight_estimate",
                                            date_method="rolling"),
    instrument_weight_estimate=estimate_block("instrument_weight_estimate",
                                              date_method="rolling"),
)

## Results

Diagnostic D is absent from the performance table because its portfolio is
not defined for the full universe; comparing a silently reduced universe
would be misleading.

In [ ]:
variants = [variant_a, variant_b, variant_c, variant_e]
aligned_variant_returns = pd.concat(
    {variant["label"]: variant["returns"] for variant in variants}, axis=1
).replace([np.inf, -np.inf], np.nan).dropna()
variant_stats = pd.DataFrame([
    R.stats_row(R.rewrap(aligned_variant_returns[variant["label"]]),
                variant["label"])
    for variant in variants
]).set_index("name")
print(f"common evaluation sample: {aligned_variant_returns.index[0].date()} "
      f"to {aligned_variant_returns.index[-1].date()}, "
      f"{len(aligned_variant_returns)} business-day observations")
variant_stats

The table is deliberately generated from one inner-aligned return frame.
Read differences as in-sample model comparisons, not proof that the top row
will win. In particular, B versus E only tests whether the default 20-year
rolling window has started discarding observations from the two weight fits;
exact equality is an empirical output, not an assumption.

The first panel below keeps each native account's realised risk. The second
rescales those **same dated PIT-gated returns** ex post to variant A's realised
volatility, so different risk levels do not masquerade as better weighting.
That normalisation is a comparison diagnostic, not another backtest.

In [ ]:
native_annual_vol = aligned_variant_returns.std() * 16
reference_vol = native_annual_vol["A fixed"]
volatility_normalised_returns = aligned_variant_returns.mul(
    reference_vol / native_annual_vol)

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
R.cumulative_from_zero(aligned_variant_returns).plot(
    ax=axes[0], title="native cumulative % return (common dates)")
axes[0].set_ylabel("percent of initial capital")
R.cumulative_from_zero(volatility_normalised_returns).plot(
    ax=axes[1],
    title=f"cumulative return at A-fixed realised volatility "
          f"({reference_vol:.1f}% annualised)")
axes[1].set_ylabel("volatility-normalised percent")
plt.tight_layout()

## What pooling does to forecast weights

Compare the estimated rule weights for a veteran (SHFE_RB, data since 2009)
and a newer market (GFEX_LC, continuous history since 2024) under pooled (B)
and unpooled diagnostic D fitting. Pooling gives the newer market the whole
universe's experience; the four panels show whether its shorter own sample
produces a materially different or less stable path. The GFEX_PD message above
records whether the latest data now contain enough annual boundaries rather
than freezing an old conclusion in prose.

In [ ]:
rb_weight_paths = pd.concat({
    "pooled (B)": variant_b["forecast_weights_rb"],
    "unpooled (D)": variant_d["forecast_weights_rb"],
}, axis=1).dropna()
lc_weight_paths = pd.concat({
    "pooled (B)": variant_b["forecast_weights_lc"],
    "unpooled (D)": variant_d["forecast_weights_lc"],
}, axis=1).dropna()

fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex="row", sharey=True)
rb_weight_paths["pooled (B)"].plot(
    ax=axes[0, 0], legend=True, title="SHFE_RB - pooled (B)")
rb_weight_paths["unpooled (D)"].plot(
    ax=axes[0, 1], legend=False, title="SHFE_RB - unpooled (D)")
lc_weight_paths["pooled (B)"].plot(
    ax=axes[1, 0], legend=False, title="GFEX_LC - pooled (B)")
lc_weight_paths["unpooled (D)"].plot(
    ax=axes[1, 1], legend=False, title="GFEX_LC - unpooled (D)")
for ax in axes.flat:
    ax.set_ylabel("forecast weight")
    ax.set_xlabel("")
plt.tight_layout()
print(f"SHFE_RB common weight sample: {rb_weight_paths.index[0].date()} to "
      f"{rb_weight_paths.index[-1].date()}")
print(f"GFEX_LC common weight sample: {lc_weight_paths.index[0].date()} to "
      f"{lc_weight_paths.index[-1].date()}")

In [ ]:
scalar_compare = pd.DataFrame({
    "fixed (production fit)": pd.Series(28.40, index=variant_b["scalar_carry60_rb"].index),
    "estimated, pooled (B)": variant_b["scalar_carry60_rb"],
    "estimated, unpooled (D)": variant_d["scalar_carry60_rb"],
}).dropna()
scalar_compare.plot(title="carry60 forecast scalar for SHFE_RB");

## Instrument weights and the diversification multipliers

Handcraft clusters the correlation matrix into nested pairs and splits
capital down the tree, then tilts by (noisy) Sharpe unless `equalise_SR`
(on by default for instrument weights). The IDM says how much the combined
portfolio can be levered because instruments diversify; it is capped at 2.5.
FDM is a different object: it diversifies trading rules within one market.
We plot them separately so two different correlation problems are not read as
one interchangeable multiplier.
The plotted weights are the native fitted weights after the dated liquidity
mask and same-date renormalisation, not the optimiser's ungated raw matrix.

In [ ]:
final_weights = variant_b["instrument_weights"].iloc[-1].sort_values(ascending=False)
largest_weights = final_weights.head(20).sort_values()
ax = largest_weights.plot.barh(figsize=(9, 7))
ax.set_title("variant B: 20 largest latest PIT-eligible estimated weights")
ax.set_xlabel("instrument weight")
ax.set_ylabel("instrument")
print(f"effective number of instruments (1/sum w^2): "
      f"{1 / (final_weights ** 2).sum():.1f}; "
      f"positive weights: {(final_weights > 0).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
pd.DataFrame({
    "IDM (B, estimated)": variant_b["idm"],
    "IDM (E, rolling weight fits)": variant_e["idm"],
}).plot(ax=ax, title="instrument diversification multiplier (IDM)")
ax.set_ylabel("IDM")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(11, 4))
variant_b["fdm_rb"].plot(
    ax=ax, color="tab:purple",
    title="SHFE_RB forecast diversification multiplier (FDM), variant B")
ax.set_ylabel("FDM")
plt.tight_layout()

## Recommendations for this universe

Judge from the tables above, but the structural argument goes:

- **Pool noisy rule evidence, not contract economics**: pooled scalars,
  correlations, gross rule returns and turnover are sensible defaults for
  young histories. Keep full cost summaries market-specific so the estimate
  is effectively own cost per trade × pooled rule turnover.
- **Do not declare an optimiser winner here**: handcraft and shrinkage were
  compared in sample. Treat either as a challenger and judge stability across
  windows; keep `equalise_SR=True` for instrument weights because notebook 05
  explains how little noisy cross-market Sharpe ranks mean.
- **Expanding versus 20-year rolling weight fits is a measured result**.
  Variant E leaves scalar and correlation estimation expanding. If its paths
  coincide with B, use expanding for simplicity; a shorter or fully rolling
  stack is a new, pre-declared adaptation experiment.
- Compare the scalar paths with the fixed external line before deciding.
  Fixing genuinely pre-existing scalars while estimating only weights and
  multipliers is a legitimate simplification that removes early-sample scalar
  noise; choosing the fixed value after seeing this graph is not.

A production-ish config can therefore start with variant B's mixed pooling
defaults and fixed external scalars, while keeping shrinkage C as a serious
challenger. Liquidity membership should remain point-in-time; a present-day
`bad_markets` list must not be projected backward over valid history.

**Next**: the pooling section below separates complete, none, and
partial/group pooling, then carries one production-ish trend+carry design
through the latest exact three-year window. The upstream manual
`docs/backtesting.md` covers the system machinery in still more depth.

## Pooling in practice

The preceding section exposed five pooling switches. That is the API, but it is not yet
a mental model. **Pooling means borrowing observations for an estimate. It
does not mean pooling capital, prices, trades, or realised P&L.**

This notebook slows down and answers the practical questions:

1. What exactly is combined by each switch?
2. What are complete pooling, no pooling, and partial/group pooling?
3. Which choices are honest in a backtest, and how do they move into
   production?
4. What did a deliberately plain 50% trend / 50% carry portfolio make over
   the latest three years in this data, net of modelled trading costs, when
   market membership is decided only from information available at the time?

The small diagnostics use four named markets and explicit code. There is no
clever loop-generating framework here: the point is to see the objects.
The final section then runs every stored Chinese history once and applies one
lagged liquidity/seasoning rule through time. A 2026 survivor list is never
projected backwards into the historical portfolio.

In [ ]:
import copy
import logging
import yaml
from IPython.utils.io import capture_output

from sysdata.config.configdata import Config
from sysdata.sim.db_futures_sim_data import dbFuturesSimData
from systems.provided.futures_chapter15.basesystem import futures_system

R.limit_blas_threads()
data = dbFuturesSimData()
universe = R.chinese_universe(data)

EWMAC = "systems.provided.rules.ewmac.ewmac"
EWMAC_DATA = ["rawdata.get_daily_prices", "rawdata.daily_returns_volatility"]
CARRY = "systems.provided.rules.carry.carry"

trading_rules = {
    "ewmac16_64": dict(function=EWMAC, data=EWMAC_DATA,
                       other_args=dict(Lfast=16, Lslow=64)),
    "ewmac32_128": dict(function=EWMAC, data=EWMAC_DATA,
                        other_args=dict(Lfast=32, Lslow=128)),
    "ewmac64_256": dict(function=EWMAC, data=EWMAC_DATA,
                        other_args=dict(Lfast=64, Lslow=256)),
    "carry10": dict(function=CARRY, data=["rawdata.raw_carry"],
                    other_args=dict(smooth_days=10)),
    "carry60": dict(function=CARRY, data=["rawdata.raw_carry"],
                    other_args=dict(smooth_days=60)),
    "carry125": dict(function=CARRY, data=["rawdata.raw_carry"],
                     other_args=dict(smooth_days=125)),
}

# Treated here as external long-run production fits. This notebook does not
# re-estimate them on the evaluation window; provenance caveat at the end.
fixed_scalars = dict(
    ewmac16_64=3.75, ewmac32_128=2.65, ewmac64_256=1.87,
    carry10=27.82, carry60=28.40, carry125=29.37,
)

# A deliberately simple prior, not an optimiser output: half the forecast
# budget to each style, equal within each style.
fixed_forecast_weights = {
    "ewmac16_64": 1 / 6, "ewmac32_128": 1 / 6, "ewmac64_256": 1 / 6,
    "carry10": 1 / 6, "carry60": 1 / 6, "carry125": 1 / 6,
}

print(f"{len(universe)} stored Chinese histories; data through "
      f"{data.daily_prices('SHFE_RB').index[-1].date()}")

## One word, three statistical operations

Those five switches implement three genuinely different kinds
of pooling:

| operation | switches | what the code does | resulting estimate |
|---|---|---|---|
| **cross-sectional reduction** | `forecast_scalar_estimate.pool_instruments` | each date, take the median absolute raw forecast across markets; then average through time | one scalar history per rule, shared by every market |
| **panel stacking** | `forecast_correlation_estimate.pool_instruments`; `forecast_weight_estimate.pool_gross_returns` | resample each market, align rule columns, add microseconds to duplicate dates, and stack market-rows vertically | more observations for rule correlations or rule P&L means/covariances |
| **summary averaging** | `forecast_cost_estimates.use_pooled_costs`; `use_pooled_turnover` | average annual SR-cost or turnover summaries across compatible markets | a steadier cost penalty used when fitting forecast weights |

There is deliberately **no master pooling switch**. A scalar can be pooled
while costs remain market-specific. Correlations can be pooled while rule
weights are fixed. That independence is useful, but it makes vague advice
like "turn pooling on" dangerous.

Two more distinctions matter:

- Pooling is only over markets with the relevant rule (and, for forecast
  weights/correlations, a compatible set of cheap rules). It is not always
  literally the entire universe.
- Instrument-weight estimation is cross-sectional by construction: its
  columns *are instruments*. It has no `pool_instruments` flag and should not
  be confused with pooling forecast evidence across instruments.

## A four-market research desk

Use three seasoned contracts from different exchanges and one young GFEX
contract. This is intentionally hand-picked for diagnosis, not claimed as a
portfolio. Rebar, soymeal and PTA tell us whether a result is shared by old
markets; lithium shows what happens when an individual history is short.

In [ ]:
FOCUS = ["SHFE_RB", "DCE_M", "CZCE_TA", "GFEX_LC"]

focus_history = pd.DataFrame({
    "SHFE_RB": [data.daily_prices("SHFE_RB").index[0],
                data.daily_prices("SHFE_RB").index[-1],
                len(data.daily_prices("SHFE_RB"))],
    "DCE_M": [data.daily_prices("DCE_M").index[0],
              data.daily_prices("DCE_M").index[-1],
              len(data.daily_prices("DCE_M"))],
    "CZCE_TA": [data.daily_prices("CZCE_TA").index[0],
                data.daily_prices("CZCE_TA").index[-1],
                len(data.daily_prices("CZCE_TA"))],
    "GFEX_LC": [data.daily_prices("GFEX_LC").index[0],
                data.daily_prices("GFEX_LC").index[-1],
                len(data.daily_prices("GFEX_LC"))],
}, index=["first", "last", "daily rows"]).T
focus_history

In [ ]:
focus_config = Config(dict(
    trading_rules=trading_rules,
    forecast_scalars=fixed_scalars,
    forecast_weights=fixed_forecast_weights,
    forecast_div_multiplier=1.5,
    instruments=FOCUS,
    instrument_weights={"SHFE_RB": 0.25, "DCE_M": 0.25,
                        "CZCE_TA": 0.25, "GFEX_LC": 0.25},
    instrument_div_multiplier=2.0,
    notional_trading_capital=100_000_000,
    base_currency="CNH",
))
focus_system = futures_system(data=dbFuturesSimData(), config=focus_config)

## 1. Forecast-scalar pooling: robust complete pooling

A raw rule has arbitrary amplitude. The scalar makes its average absolute
forecast about 10. With complete pooling the implementation first takes the
cross-sectional median absolute forecast on each date, then a rolling time
average. The median stops one wild market dominating and taking the
cross-section first avoids jumps merely because another market was listed.

Below we reproduce the actual scalar function four ways. "Veteran pool" is
**group pooling**: lithium borrows the three old markets' scale rather than
either standing alone or entering the full pool. It is a coarse practical
substitute for true partial pooling, which would blend own, group and global
estimates continuously. This grouping is a research choice; there is no
asset-class/group switch for scalars in the standard config.

In [ ]:
from sysquant.estimators.forecast_scalar import forecast_scalar

rb_raw = focus_system.forecastScaleCap.get_raw_forecast(
    "SHFE_RB", "carry60").squeeze()
m_raw = focus_system.forecastScaleCap.get_raw_forecast(
    "DCE_M", "carry60").squeeze()
ta_raw = focus_system.forecastScaleCap.get_raw_forecast(
    "CZCE_TA", "carry60").squeeze()
lc_raw = focus_system.forecastScaleCap.get_raw_forecast(
    "GFEX_LC", "carry60").squeeze()

raw_carry_panel = pd.concat(
    [rb_raw.rename("SHFE_RB"), m_raw.rename("DCE_M"),
     ta_raw.rename("CZCE_TA"), lc_raw.rename("GFEX_LC")], axis=1)

scalar_complete = forecast_scalar(raw_carry_panel, min_periods=500,
                                  backfill=False).rename("complete: four markets")
scalar_veterans = forecast_scalar(raw_carry_panel[
    ["SHFE_RB", "DCE_M", "CZCE_TA"]], min_periods=500,
    backfill=False).rename("group: veterans")
scalar_rb = forecast_scalar(raw_carry_panel[["SHFE_RB"]], min_periods=500,
                            backfill=False).rename("none: rebar alone")
scalar_lc = forecast_scalar(raw_carry_panel[["GFEX_LC"]], min_periods=500,
                            backfill=False).rename("none: lithium alone")

scalar_demo = pd.concat(
    [scalar_complete, scalar_veterans, scalar_rb, scalar_lc], axis=1)
print("latest scalar estimate:")
display(scalar_demo.apply(lambda series: series.dropna().iloc[-1]).to_frame("scalar"))
scalar_demo.loc["2023":].plot(title="carry60 scalar: what each market is allowed to learn");

Read this as a bias–variance choice, not a contest with a universal winner:

- **No pooling** preserves a genuinely different market but gives lithium
  only its short and regime-specific sample.
- **Complete pooling** is stable and gives a new listing a usable estimate on
  day one, but assumes rule amplitude is exchangeable across markets.
- **Group pooling** is often a useful answer when asset classes differ. True
  **partial pooling** blends own and shared evidence according to uncertainty.
  In this codebase either normally means separate systems, separate pre-fit
  constants, or a custom estimator—not another Boolean.

The default scalar estimator also has `backfill=True`: it fills the early
period with the first estimate, which uses later data. That small, explicit
look-ahead is convenient for long backtests but cannot be recreated live.
Fixed long-run scalars, used in the final run, are the cleaner production
choice when those constants were chosen outside the test window.

## 2. Correlation pooling: stack markets, keep rules as columns

For forecast correlations, time remains horizontal and rules remain columns;
markets supply additional rows. The real estimator does this through time
with expanding or rolling fits and adjusts exponential lookbacks for the
number of stacked markets. This cell exposes the same stacking mechanism on
our four markets and reports three economically meaningful pairs.

Pooling destroys cross-market ordering at equal timestamps (the code offsets
rows by microseconds), which is fine for contemporaneous rule correlation but
would be wrong for estimating lead/lag or autocorrelation.

In [ ]:
from syscore.pandas.list_of_df import (
    listOfDataFrames, stacked_df_with_added_time_from_list,
)

rb_weekly = focus_system.combForecast.get_all_forecasts("SHFE_RB").resample("W").last()
m_weekly = focus_system.combForecast.get_all_forecasts("DCE_M").resample("W").last()
ta_weekly = focus_system.combForecast.get_all_forecasts("CZCE_TA").resample("W").last()
lc_weekly = focus_system.combForecast.get_all_forecasts("GFEX_LC").resample("W").last()

pooled_weekly = stacked_df_with_added_time_from_list(listOfDataFrames([
    rb_weekly, m_weekly, ta_weekly, lc_weekly,
]))

rb_corr = rb_weekly.corr(min_periods=20)
lc_corr = lc_weekly.corr(min_periods=20)
pooled_corr = pooled_weekly.corr(min_periods=20)

correlation_demo = pd.DataFrame({
    "rebar only": [
        rb_corr.loc["carry60", "carry125"],
        rb_corr.loc["ewmac32_128", "ewmac64_256"],
        rb_corr.loc["carry60", "ewmac32_128"],
    ],
    "lithium only": [
        lc_corr.loc["carry60", "carry125"],
        lc_corr.loc["ewmac32_128", "ewmac64_256"],
        lc_corr.loc["carry60", "ewmac32_128"],
    ],
    "four-market pool": [
        pooled_corr.loc["carry60", "carry125"],
        pooled_corr.loc["ewmac32_128", "ewmac64_256"],
        pooled_corr.loc["carry60", "ewmac32_128"],
    ],
}, index=["carry60 vs carry125", "trend32 vs trend64",
          "carry60 vs trend32"])

print("weekly rows before dropping missing values:")
display(pd.Series({"rebar": len(rb_weekly), "lithium": len(lc_weekly),
                   "stacked four-market panel": len(pooled_weekly)},
                  name="rows").to_frame())
correlation_demo

## 3. Return pooling is not forecast pooling

`forecast_weight_estimate.pool_gross_returns` stacks **hypothetical weekly
P&L from each rule**, not the forecast values above. The optimiser then learns
rule means, volatilities and correlations from that P&L panel. This answers
"which rules have paid?"; scalar and forecast-correlation pooling answer
different questions.

Subtleties worth remembering:

- With `pool_gross_returns=True`, gross evidence is shared. If costs remain
  market-specific, fitted weights can still differ by market because their
  net evidence differs.
- `equalise_SR`, shrinkage and pooling are separate controls. Shrinkage pulls
  noisy rule estimates toward priors **after** the sample is assembled; it is
  not hierarchical partial pooling across instruments.
- An expanding fit uses only history available before each use period. An
  `in_sample` fit sees the future and belongs only in a diagnostic.
- A newborn market cannot create annual unpooled fitting periods. Complete
  pooling is what lets it inherit a usable rule mix.

## 4. Cost pooling and turnover pooling

Costs have two pieces: how much a rule trades, and what one unit of turnover
costs in a particular contract. Pooling turnover says "this rule usually
trades about this much" while preserving rebar's or lithium's own price,
volatility, point value and spread cost. Fully pooled cost says the whole
annual SR-cost number is exchangeable too.

The middle choice—**own contract cost × pooled rule turnover**—is usually
the sensible production default. Turnover is a property of the rule and is
noisy in a short series; cost per trade really is market-specific.

In [ ]:
def focus_system_with_cost_pooling(use_pooled_costs, use_pooled_turnover):
    return futures_system(data=dbFuturesSimData(), config=Config(dict(
        trading_rules=trading_rules,
        forecast_scalars=fixed_scalars,
        forecast_weights=fixed_forecast_weights,
        instruments=FOCUS,
        instrument_weights={"SHFE_RB": 0.25, "DCE_M": 0.25,
                            "CZCE_TA": 0.25, "GFEX_LC": 0.25},
        notional_trading_capital=100_000_000,
        base_currency="CNH",
        forecast_cost_estimates=dict(
            use_pooled_costs=use_pooled_costs,
            use_pooled_turnover=use_pooled_turnover,
        ),
    )))

cost_own = focus_system_with_cost_pooling(False, False)
turnover_pool = focus_system_with_cost_pooling(False, True)
cost_pool = focus_system_with_cost_pooling(True, False)

cost_pooling_demo = pd.DataFrame({
    "SHFE_RB carry60": [
        cost_own.accounts.forecast_turnover("SHFE_RB", "carry60"),
        turnover_pool.accounts.forecast_turnover("SHFE_RB", "carry60"),
        cost_own.accounts.get_SR_transaction_cost_for_instrument_forecast(
            "SHFE_RB", "carry60"),
        turnover_pool.accounts.get_SR_transaction_cost_for_instrument_forecast(
            "SHFE_RB", "carry60"),
        cost_pool.accounts.get_SR_transaction_cost_for_instrument_forecast(
            "SHFE_RB", "carry60"),
    ],
    "GFEX_LC carry60": [
        cost_own.accounts.forecast_turnover("GFEX_LC", "carry60"),
        turnover_pool.accounts.forecast_turnover("GFEX_LC", "carry60"),
        cost_own.accounts.get_SR_transaction_cost_for_instrument_forecast(
            "GFEX_LC", "carry60"),
        turnover_pool.accounts.get_SR_transaction_cost_for_instrument_forecast(
            "GFEX_LC", "carry60"),
        cost_pool.accounts.get_SR_transaction_cost_for_instrument_forecast(
            "GFEX_LC", "carry60"),
    ],
}, index=["own turnover", "four-market pooled turnover",
          "own transaction SR cost", "own cost x pooled turnover",
          "fully pooled transaction SR cost"])
cost_pooling_demo

These cost switches govern the **synthetic SR-cost penalty used to choose
forecast weights and reject expensive rules**. They do not replace final
portfolio accounting: the system's realised net P&L still charges each
instrument's actual configured cash/spread costs to its own trades. If
forecast weights are fixed, as in the final run, changing these estimation
switches does not change those weights or the final cash-cost calculation.

## Complete, none, or partial? A decision rule

| situation | scalar | rule correlation | rule gross P&L | turnover | cost per trade |
|---|---|---|---|---|---|
| young, broad Chinese universe | complete pool | complete pool | complete pool if estimating weights | pool | own market |
| mature and demonstrably heterogeneous asset classes | group/partial pool | group/partial pool | group/partial pool | usually pool by rule | own market |
| one market with decades of data and a structural reason to differ | perhaps own | perhaps own | own | pooled as a prior | own market |
| production simplicity / weak evidence for fitted means | fixed external scalar | pool for FDM | **fixed rule weights** | irrelevant to fixed weights | actual cash cost |

Partial pooling is not available as a universal config value. Practical
implementations are:

1. fit separate groups (for example financials, commodities) and store the
   resulting constants;
2. blend an own estimate with the pooled estimate according to effective
   sample size; or
3. write a hierarchical estimator.

Do not silently call an asset-class average "more realistic" after seeing
which grouping wins. The grouping and blend rule must be chosen before the
evaluation window.

## Backtest-to-production protocol

The safest way to think about a fit is **fit through date _t_, freeze it,
trade after _t_**.

In a backtest:

- use `date_method="expanding"` or a pre-declared rolling window;
- build a fresh system for every configuration (cached estimates otherwise
  leak between experiments);
- never use `in_sample` for a performance claim;
- keep universe/liquidity eligibility point-in-time; and
- record whether early scalar backfill is present.

In production:

- refit offline on a schedule (annual is plenty for these slow estimates),
  version the fitted artefact and deploy it for the next period;
- let a new market borrow pooled scalars/correlations/turnover, but require
  enough own price/volatility history and liquidity before allocating risk;
- monitor realised turnover, forecast amplitude and correlations against the
  fitted panel; and
- retain caps, smoothing and a fallback set of fixed weights. "Re-estimate
  every day" is not the same thing as "adapt intelligently".

## The actual money question: a production-ish 50/50 model

Now make one choice and live with it. This specification is intentionally
boring:

- three medium/slow EWMAC rules and three carry smoothings;
- 50% of forecast weight to trend, 50% to carry, equal within style;
- fixed external forecast scalars (no early backfill and no refit here);
- pooled, expanding forecast correlations for the FDM;
- expanding handcraft instrument weights with `equalise_SR=True`, so noisy
  per-market mean returns do not decide the allocation;
- estimated IDM, 10% forecast buffering, delayed fills, whole contracts and
  each market's configured native cash/spread fill costs.

The full history is run first and only then sliced. Starting the engine three
years ago would give it no mature volatility, correlation, weight or buffer
state and would answer a different question.

Membership is also historical. A market enters only after its held contract
has averaged at least 130 contracts over 20 observed sessions, remains until
that trailing mean falls below 70, and uses the normal one-business-row delay.
Around exchange holidays this is an upstream execution approximation, not an
exchange-calendar-accurate next-session fill. Known
terminal histories are targeted flat before their final stored close so the
exit and its native transaction cost are present in P&L. That terminal close
is an explicitly ex-post data-boundary assumption, not a causal delisting
signal. The volume thresholds themselves are deliberately modest tutorial
choices, not a capacity claim.

For this point-in-time study, `vol_normalise_currency_costs=False` is also
pre-declared. The optional upstream rescaling divides all historical costs by
the instrument's **final** 180-day volatility. That is future-informed and can
be zero for a terminal history. We retain native cash/spread fill costs and
whole-contract rounding; we only remove that ex-post cost deflator. This is a
research choice, not the upstream default.

In [ ]:
from systems.accounts.accounts_stage import Account
from systems.basesystem import System
from systems.forecast_combine import ForecastCombine
from systems.forecast_scale_cap import ForecastScaleCap
from systems.forecasting import Rules
from systems.positionsizing import PositionSizing
from systems.rawdata import RawData


print("reading held-contract volume for the historical universe ...")
with capture_output():
    held_volumes = R.held_contract_volumes(data, universe)
liquidity = R.liquidity_eligibility(held_volumes, force_terminal_close=True)
liquidity_events = R.liquidity_event_table(liquidity, held_volumes)


def point_in_time_futures_system(data, config, eligibility,
                                 fixed_weights=None):
    '''The normal futures system with only its portfolio stage replaced.'''
    return System(
        [
            Account(),
            R.PointInTimePortfolios(
                eligibility=eligibility, fixed_weights=fixed_weights),
            PositionSizing(), RawData(), ForecastCombine(),
            ForecastScaleCap(), Rules(),
        ],
        data,
        config,
    )


eligibility_summary = pd.DataFrame({
    "first eligible": {
        code: liquidity.index[liquidity[code]][0].date()
        if liquidity[code].any() else pd.NaT
        for code in universe
    },
    "last eligible": {
        code: liquidity.index[liquidity[code]][-1].date()
        if liquidity[code].any() else pd.NaT
        for code in universe
    },
    "eligible business days": liquidity.sum(),
})
print(f"{int((liquidity.sum() > 0).sum())} histories ever pass the rule; "
      f"{int(liquidity.iloc[-1].sum())} pass on {liquidity.index[-1].date()}")
display(eligibility_summary.sort_values("last eligible").head(12))
display(liquidity_events.tail(12))


with open(R.REPO_ROOT / "sysdata/config/defaults.yaml") as handle:
    DEFAULTS = yaml.safe_load(handle)

pooled_forecast_corr = copy.deepcopy(DEFAULTS["forecast_correlation_estimate"])
pooled_forecast_corr.update(pool_instruments=True, date_method="expanding")

instrument_weight_fit = copy.deepcopy(DEFAULTS["instrument_weight_estimate"])
instrument_weight_fit.update(method="handcraft", equalise_SR=True,
                             date_method="expanding")

instrument_corr_fit = copy.deepcopy(DEFAULTS["instrument_correlation_estimate"])
instrument_corr_fit.update(date_method="expanding")

productionish_config = Config(dict(
    trading_rules=trading_rules,
    forecast_scalars=fixed_scalars,
    forecast_weights=fixed_forecast_weights,
    use_forecast_scale_estimates=False,
    use_forecast_weight_estimates=False,
    use_forecast_div_mult_estimates=True,
    forecast_correlation_estimate=pooled_forecast_corr,
    use_instrument_weight_estimates=True,
    instrument_weight_estimate=instrument_weight_fit,
    use_instrument_div_mult_estimates=True,
    instrument_correlation_estimate=instrument_corr_fit,
    instruments=universe,              # do not also set instrument_weights
    notional_trading_capital=100_000_000,
    percentage_vol_target=16.0,
    base_currency="CNH",
    vol_normalise_currency_costs=False,
    capital_multiplier=dict(func="syscore.capital.fixed_capital"),
))

productionish_system = point_in_time_futures_system(
    data=dbFuturesSimData(), config=productionish_config,
    eligibility=liquidity)

# Estimation progress bars are valuable interactively but make a terrible
# saved notebook. The result cells below are the research record.
with capture_output():
    portfolio = productionish_system.accounts.portfolio()
    production_weights = productionish_system.portfolio.get_instrument_weights()

allowed_on_weight_dates = liquidity.reindex(production_weights.index).ffill()
allowed_on_weight_dates = allowed_on_weight_dates.reindex(
    columns=production_weights.columns, fill_value=False).fillna(False)
assert (production_weights.where(~allowed_on_weight_dates, 0.0).abs().max().max()
        < 1e-12)

print("full-history point-in-time run complete")

In [ ]:
gross = portfolio.percent.gross.as_ts.rename("gross")
net = portfolio.percent.as_ts.rename("modelled net")
costs = portfolio.percent.costs.as_ts.rename("costs")

# One accounting frame is the source of every number and graph below.  Do not
# let pandas silently compare gross, net and costs on different dates.
accounting = pd.concat([gross, net, costs], axis=1)
accounting = accounting.replace([np.inf, -np.inf], np.nan).dropna()
reconciliation_error = (
    accounting["modelled net"]
    - accounting["gross"]
    - accounting["costs"]
).abs()
assert reconciliation_error.max() < 1e-8

data_end = accounting.index.max()
three_year_cutoff = data_end - pd.DateOffset(years=3)

# Strictly after the cutoff gives 2023-07-28 through 2026-07-27 here.
recent = accounting.loc[accounting.index > three_year_cutoff].copy()
recent["2x cost stress"] = recent["gross"] + 2 * recent["costs"]

def fixed_capital_summary(series):
    series = series.dropna()
    curve = R.rewrap(series)
    return dict(
        observations=len(series),
        first=series.index[0].date(),
        last=series.index[-1].date(),
        total_return=series.sum(),
        ann_mean=curve.percent.ann_mean(),
        ann_vol=curve.percent.ann_std(),
        sharpe=curve.percent.sharpe(),
        worst_drawdown=curve.percent.worst_drawdown(),
    )

recent_summary = pd.DataFrame({
    "gross": fixed_capital_summary(recent["gross"]),
    "modelled net": fixed_capital_summary(recent["modelled net"]),
    "2x cost stress": fixed_capital_summary(recent["2x cost stress"]),
}).T
print(f"maximum |net - gross - costs|: {reconciliation_error.max():.3g}")
recent_summary

`total_return` above is the direct pysystemtrade answer: percentage P&L on
**fixed 100m CNH notional capital**. Fixed capital is useful because risk does
not snowball during a research comparison. Multiplying by 1m converts one
percentage point into CNH for this configuration.

A client who continuously reinvested would experience compounded wealth
instead. Mechanically compounding the same daily percentage stream is a
useful illustration, but it is not a second independent backtest: exact
variable-capital positions would differ slightly because contracts are
integer-sized and buffers create path dependence.

In [ ]:
net_fixed_total = recent["modelled net"].sum()
net_compounded_total = (
    (1 + recent["modelled net"] / 100).prod() - 1
) * 100
net_cnh = net_fixed_total / 100 * 100_000_000

pd.Series({
    "fixed-capital net return (%)": net_fixed_total,
    "fixed-capital net P&L (CNH)": net_cnh,
    "mechanically reinvested wealth return (%)": net_compounded_total,
}, name="latest exact three-year window").to_frame()

In [ ]:
R.cumulative_from_zero(
    recent[["gross", "modelled net", "2x cost stress"]]
).plot(
    title=f"latest three years: cumulative % of fixed capital "
          f"({recent.index[0].date()} to {recent.index[-1].date()})")

calendar_returns = recent[["gross", "modelled net", "2x cost stress"]].groupby(
    recent.index.year).sum()
calendar_returns.index.name = "calendar year (2023 and 2026 are partial)"
calendar_returns

## How much confidence should we put in three years?

The next cell writes the headline from the same aligned frame as the table and
chart. That prevents an old sentence surviving after the data are refreshed.

In [ ]:
net_row = recent_summary.loc["modelled net"]
gross_row = recent_summary.loc["gross"]
stress_row = recent_summary.loc["2x cost stress"]
largest_year = calendar_returns["modelled net"].abs().idxmax()
largest_year_return = calendar_returns.loc[largest_year, "modelled net"]
largest_year_share = (
    100 * largest_year_return / net_fixed_total
    if net_fixed_total != 0 else np.nan
)

print(
    f"Exact window: {net_row['first']} through {net_row['last']} "
    f"({int(net_row['observations'])} aligned observations)."
)
print(
    f"Fixed-capital net return {net_row['total_return']:+.2f}%; "
    f"annual mean {net_row['ann_mean']:.2f}%, annual volatility "
    f"{net_row['ann_vol']:.2f}%, Sharpe {net_row['sharpe']:.3f}, "
    f"worst arithmetic drawdown {net_row['worst_drawdown']:.2f}% "
    "of initial capital."
)
print(
    f"Gross {gross_row['total_return']:+.2f}%; two-times-cost stress "
    f"{stress_row['total_return']:+.2f}%; mechanical reinvestment "
    f"{net_compounded_total:+.2f}%."
)
print(
    f"Largest absolute calendar contribution: {largest_year}, "
    f"{largest_year_return:+.2f} percentage points "
    f"({largest_year_share:.1f}% of the window total; boundary years may be partial)."
)

Mechanical reinvestment is not the native fixed-risk backtest result, and a
large contribution from one calendar year is evidence against reading the
three-year total as three uniform years of edge.

This is a useful implementation result, not an expected-return forecast.
Three years contain only one inflation/commodity/policy path, and the
configuration was not run as a sealed prospective experiment. In particular:

- fixed rule budgets prevent us from choosing carry merely because it won
  recently, and every fitted correlation/weight used an expanding window;
- nevertheless the long-run scalar constants were developed elsewhere, not
  registered in a pre-study protocol;
- membership comes from the declared lagged point-in-time liquidity and
  seasoning rule. Its thresholds are research assumptions, not proof that
  every indicated order could have been filled; and
- capacity, exchange limit moves, margin calls, tax and operational failures
  are outside this daily simulation.

The most honest headline is therefore the modelled fixed-capital result with
its volatility and drawdown beside it, plus the 2x-cost stress—not the
compounded number alone.

## Production recommendation

For this young Chinese universe, pool rule amplitude, rule correlations and
rule turnover unless a pre-declared group test demonstrates stable
heterogeneity. Keep cost per trade market-specific. Avoid fitting forecast
means unless the result is stable across windows and optimiser families;
fixed 50/50 style budgets are a very respectable production prior. Refit the
slow covariance/weight objects offline, version them, and make the next
year—not the fitting history—the scorecard.

**Next**: notebook 05 compares carry and trend under honest exclusions;
notebook 06 and the compact lab compare alternative trend-rule families.